<a href="https://colab.research.google.com/github/Vidushi0907/Machine-Learning-/blob/main/Case_Study_3_Amazon_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 3: Sentiment Analysis on Amazon Product Reviews
**MLE Exam | LSTM Baseline vs Fine-tuned BERT**

**Aim:** Classify Amazon product reviews as negative, neutral, or positive, compare an LSTM baseline with a pre-trained BERT model, and discuss transfer learning and deployment constraints.

**Dataset (Kaggle):** [Amazon Fine Food Reviews](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

## How to use in Google Colab
1. Open this notebook in Colab.
2. Select **Runtime → Change runtime type → T4 GPU** if available (recommended for BERT).
3. Run cells from top to bottom.
4. When prompted, upload `Reviews.csv` from the Kaggle dataset ZIP.

This notebook uses a balanced sample of 3,000 reviews (1,000 per class) to keep the practical experiment manageable. The original dataset has an imbalanced sentiment distribution, so results on this balanced sample may differ from real-world performance.

## 1. Install required libraries

In [ ]:
!pip -q install transformers accelerate

## 2. Upload and load the Kaggle dataset

In [ ]:
from google.colab import files
import os

if not os.path.exists("/content/Reviews.csv"):
    print("Upload Reviews.csv (extract it from the Kaggle dataset ZIP first).")
    uploaded = files.upload()

import pandas as pd

df = pd.read_csv("/content/Reviews.csv")
print("Dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())
df.head()

## 3. Prepare the data
Rating mapping: 1–2 stars = Negative (0), 3 stars = Neutral (1), 4–5 stars = Positive (2).

We sample 1,000 reviews per class and use a stratified train/test split. The test set is held out from model training.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Keep only required columns and remove missing values
df = df[["Text", "Score"]].dropna()
df["Text"] = df["Text"].astype(str)

def get_sentiment(score):
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df["Sentiment"] = df["Score"].apply(get_sentiment)

# Select 1,000 examples from each sentiment class
df = df.groupby("Sentiment", group_keys=False).sample(
    n=1000, random_state=42
).reset_index(drop=True)

X = df["Text"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nClass counts in sampled data:")
print(df["Sentiment"].value_counts().sort_index().rename(
    index={0: "Negative", 1: "Neutral", 2: "Positive"}
))

## 4. Model 1 — LSTM baseline

An LSTM is a type of recurrent neural network that learns patterns in sequences. Here, words are converted into numbers, padded to a fixed length, and passed into an LSTM classifier.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Make results more reproducible
np.random.seed(42)
tf.random.set_seed(42)

# Convert text into token sequences
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Make all reviews length 100
X_train_pad = pad_sequences(
    X_train_seq, maxlen=100, padding="post", truncating="post"
)
X_test_pad = pad_sequences(
    X_test_seq, maxlen=100, padding="post", truncating="post"
)

y_train_np = np.array(y_train)
y_test_np = np.array(y_test)

# Build the LSTM model
model_lstm = Sequential([
    Embedding(input_dim=10000, output_dim=64),
    LSTM(64),
    Dense(3, activation="softmax")
])

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_lstm.summary()

In [ ]:
history_lstm = model_lstm.fit(
    X_train_pad,
    y_train_np,
    epochs=3,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

lstm_loss, lstm_accuracy = model_lstm.evaluate(
    X_test_pad, y_test_np, verbose=0
)
y_pred_lstm = np.argmax(model_lstm.predict(X_test_pad, verbose=0), axis=1)

print("LSTM test accuracy:", round(lstm_accuracy, 4))
print("\nLSTM classification report:")
print(classification_report(
    y_test_np, y_pred_lstm,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

## 5. Model 2 — Fine-tune pre-trained BERT
BERT has already learned language patterns from large text collections. Fine-tuning adapts it to our three sentiment classes. This section may take several minutes and is much faster with a GPU.

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize reviews for BERT
train_encodings = bert_tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)
test_encodings = bert_tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }
        item["labels"] = torch.tensor(
            int(self.labels[index]), dtype=torch.long
        )
        return item

train_dataset = ReviewDataset(train_encodings, y_train.tolist())
test_dataset = ReviewDataset(test_encodings, y_test.tolist())

model_bert = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3
)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Fine-tune BERT
The evaluation dataset below is the held-out test set. For a strict research workflow, use a separate validation set to choose training settings and keep the test set for final evaluation.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Use compatible argument name across Transformers versions
import inspect
training_args_parameters = inspect.signature(TrainingArguments.__init__).parameters
eval_argument = (
    {"eval_strategy": "epoch"}
    if "eval_strategy" in training_args_parameters
    else {"evaluation_strategy": "epoch"}
)

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    **eval_argument
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
bert_results = trainer.evaluate()
print("BERT evaluation results:")
print(bert_results)

bert_output = trainer.predict(test_dataset)
y_pred_bert = np.argmax(bert_output.predictions, axis=1)

print("\nBERT classification report:")
print(classification_report(
    y_test, y_pred_bert,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

## 6. Compare LSTM and BERT

In [ ]:
lstm_test_accuracy = accuracy_score(y_test_np, y_pred_lstm)
bert_test_accuracy = accuracy_score(y_test, y_pred_bert)

comparison = pd.DataFrame({
    "Model": ["LSTM", "BERT"],
    "Test Accuracy": [lstm_test_accuracy, bert_test_accuracy]
})
comparison["Test Accuracy (%)"] = (comparison["Test Accuracy"] * 100).round(2)

display(comparison)

if bert_test_accuracy > lstm_test_accuracy:
    print("BERT achieved higher accuracy on this run.")
elif lstm_test_accuracy > bert_test_accuracy:
    print("LSTM achieved higher accuracy on this run.")
else:
    print("Both models achieved the same accuracy on this run.")

## 7. Predict sentiment for a new review
This example uses the fine-tuned BERT model.

In [ ]:
def predict_sentiment(review):
    inputs = bert_tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Send inputs to the same device as the model
    device = next(model_bert.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    model_bert.eval()
    with torch.no_grad():
        outputs = model_bert(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)[0]
    label = int(torch.argmax(probabilities).item())

    names = {0: "Negative", 1: "Neutral", 2: "Positive"}
    return names[label], probabilities.cpu().numpy()

review = "The product is excellent and works exactly as expected!"
sentiment, probabilities = predict_sentiment(review)

print("Review:", review)
print("Predicted sentiment:", sentiment)
print("Probabilities (Negative, Neutral, Positive):", np.round(probabilities, 3))

## 8. Transfer learning and deployment discussion

### Transfer learning efficiency
- BERT reuses language representations learned during pre-training.
- Fine-tuning adapts those representations to Amazon review sentiment.
- It often performs strongly with less task-specific data than training a language model from scratch.
- Fine-tuning still needs time, memory, and compute; it is not free.

### Real-world constraints
- **Latency:** BERT may take longer per prediction than a smaller model.
- **Memory and cost:** BERT has a larger resource footprint; consider batching, quantization, or distillation.
- **Long reviews:** Text beyond the token limit is truncated in this notebook.
- **Class imbalance:** Real customer reviews may not have equal numbers of each sentiment. Evaluate per-class precision, recall, and F1.
- **Drift:** New products, slang, and customer behavior may change performance over time.
- **Privacy:** Handle customer data under appropriate privacy and security controls.

### Conclusion
This experiment compares an LSTM baseline with a fine-tuned pre-trained BERT model on three-class Amazon review sentiment classification. Compare their measured accuracy and classification reports rather than assuming one must always win. BERT can benefit from transfer learning and contextual representations, while LSTM can be lighter. Deployment decisions should consider accuracy, per-class performance, latency, memory, cost, and the natural class distribution.

## 9. Viva questions
1. **What is sentiment analysis?** Identifying opinions or emotional tone in text.
2. **What is BERT?** A transformer-based pre-trained language model.
3. **What is fine-tuning?** Further training a pre-trained model on a task-specific dataset.
4. **What is transfer learning?** Reusing knowledge learned on one task for another task.
5. **Why use LSTM as a baseline?** It learns sequential patterns in text and provides a useful comparison.
6. **Why convert ratings to three classes?** To frame the task as negative, neutral, and positive classification.
7. **What is precision?** Of the items predicted as a class, the fraction that truly belong to it.
8. **What is recall?** Of all actual items in a class, the fraction correctly identified.
9. **Why use macro F1?** It gives each class equal weight, which is useful when class distributions differ.
10. **What is a limitation of BERT?** It can require more compute and memory than smaller models.

**Exam reminder:** Run the notebook once before your practical. BERT fine-tuning may be slow without a GPU. The exact scores will vary with data sampling, library versions, and training conditions.